# Logistic Regression — your first classifier

Companion notebook for the **Logistic Regression** deck.

You already know linear regression: `X`, `y`, `fit`, `predict`, `score`. Today the target stops being
a number and becomes a category, and three things change:

1. the model — $\hat p = \sigma(wx + b)$, a probability, not a raw number
2. the loss — **cross-entropy**, not MSE, and it has **no closed form** — gradient descent is the whole story
3. the workflow — same **LOAD &rarr; SPLIT &rarr; FIT &rarr; PREDICT &rarr; EVALUATE**, plus `stratify` and `predict_proba`

Cells marked `TODO` are yours to fill in. Solutions are at the bottom — try first.


## 0 · Setup

The dataset is embedded below, so this notebook runs anywhere with no file upload and no internet.


In [ ]:
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CSV = """hours_studied,passed
0.5,0
0.5,0
0.6,0
0.8,0
0.9,0
1.0,0
1.3,0
1.9,0
1.9,0
1.9,0
2.2,0
2.3,0
2.4,0
2.5,0
2.7,0
2.8,0
2.9,0
3.0,0
3.2,0
3.2,0
3.4,0
3.8,0
3.8,0
3.9,0
4.0,0
4.5,0
4.7,0
4.7,1
5.0,1
5.0,1
5.1,1
5.1,0
5.1,1
5.1,0
5.4,1
5.5,0
5.8,1
5.9,1
6.0,1
6.1,1
6.1,0
6.2,1
6.3,1
6.7,1
7.2,1
7.5,1
7.6,1
7.7,1
7.8,1
7.9,1
8.0,1
8.1,1
8.3,1
8.4,1
8.4,1
8.6,1
8.8,1
9.3,1
9.4,1
9.5,1"""

df = pd.read_csv(io.StringIO(CSV))
print(df.shape)
df.head()


> In the deck this was `pd.read_csv('data/study_hours_pass.csv')`. Same 60 rows — only the source differs.

**The data:** 60 students. `hours_studied` before an exam, and whether they `passed` — 0 or 1, not a scale.


## 1 · Look before you model — differently now

**TODO 1** — print `df['passed'].value_counts()` and `df['hours_studied'].describe()`.
Why is `value_counts()` the right call for `passed` but `describe()` would be misleading on it?


In [ ]:
# TODO 1


**TODO 2** — scatter plot: `hours_studied` on x, `passed` on y.

Where do the two rows of dots overlap? That overlap is *why* we need a probability, not a promise.


In [ ]:
# TODO 2


## 2 · Build X and y

Same rule as linear regression — `X` is always 2-D, `y` is always 1-D. The target being 0/1 instead of a
scale changes nothing about the shapes.

**TODO 3** — build `X` and `y` and print both shapes. Expect `(60, 1)` and `(60,)`.


In [ ]:
# TODO 3
X = ...
y = ...

print(X.shape, y.shape)


## 3 · Split — now with `stratify`

Classes can get unlucky in a plain random split: one bad seed can hand the test set 9 fails and only 3
passes, nowhere near the true ~50/50 balance. `stratify=y` prevents that.

**TODO 4** — split 80/20 with `random_state=42` and `stratify=y`. Print the four shapes, then
`y_test.value_counts()` to confirm the test set is balanced.


In [ ]:
from sklearn.model_selection import train_test_split

# TODO 4
X_train, X_test, y_train, y_test = ...

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)


## 4 · Fit

**TODO 5** — create a `LogisticRegression(max_iter=1000)`, fit it on the **training** data, then print
`coef_` and `intercept_`.

Why `max_iter=1000`? The solver is *iterative* this time — there's no algebra shortcut like linear
regression had. The default cap (100) is often too low and triggers a `ConvergenceWarning`.


In [ ]:
from sklearn.linear_model import LogisticRegression

# TODO 5
model = ...

print('w =', model.coef_)
print('b =', model.intercept_)


**TODO 6** — compute the decision boundary in hours: `-intercept_ / coef_`. Write a sentence:

> A student who studies more than **___** hours is predicted to pass.


In [ ]:
# TODO 6


## 5 · Predict — `predict` vs `predict_proba`

**TODO 7** — for a student who studied **6.5 hours**, print both `model.predict_proba([[6.5]])` and
`model.predict([[6.5]])`. Then predict the whole test set into `preds` using `predict`.

Remember the 2-D input rule: `[[6.5]]`, not `[6.5]` — same trap as linear regression.


In [ ]:
# TODO 7
preds = ...


## 6 · Evaluate

**TODO 8** — compute `accuracy_score(y_test, preds)`.

**TODO 9** — also compute accuracy on the **training** set (`model.score(X_train, y_train)`) and compare.
Which is higher? Is that surprising?


In [ ]:
from sklearn.metrics import accuracy_score

# TODO 8


In [ ]:
# TODO 9


## 7 · See the fitted curve

**TODO 10** — scatter all 60 points, then draw the fitted **sigmoid** on top using `model.predict_proba`.

Hint: `xs = np.linspace(0, 10, 200).reshape(-1, 1)`, then plot `xs` against
`model.predict_proba(xs)[:, 1]` — column 1 is `P(passed)`.


In [ ]:
# TODO 10
# hint: xs = np.linspace(0, 10, 200).reshape(-1, 1)
#       plt.plot(xs, model.predict_proba(xs)[:, 1])


## 8 · Prove sklearn is not magic

Unlike linear regression, there is **no closed-form formula** here — only gradient descent. From the deck
(Part IV), the gradient is:

$$\frac{\partial J}{\partial w} = \frac{1}{n}\sum_i (p_i - y_i)x_i, \qquad \frac{\partial J}{\partial b} = \frac{1}{n}\sum_i (p_i - y_i)$$

**TODO 11** — implement full-batch gradient descent in NumPy on **all 60 rows**: start at `w=0, b=0`,
take 20,000 steps with `lr=0.3`. You should land near `w=2.2124, b=-11.1581`.

Then fit `LogisticRegression()` (**defaults**, no `max_iter` override) on the same 60 rows and compare —
they will *not* match. Why not? (Hint: read about the `C` parameter.)


In [ ]:
x_all = df['hours_studied'].values.astype(float)
y_all = df['passed'].values.astype(float)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# TODO 11
w, b = 0.0, 0.0
lr = 0.3
for _ in range(20_000):
    pass

print(w, b)


## 9 · Does `stratify` actually matter?

**TODO 12** — loop `random_state` in `[0, 1, 2, 3, 4]` **twice**: once calling `train_test_split` with
`stratify=y`, once without. For each, print the test accuracy *and* `np.bincount(y_test)`.

Does the stratified version's class balance ever wobble? Does the unstratified one's accuracy swing more?


In [ ]:
# TODO 12
for rs in [0, 1, 2, 3, 4]:
    pass


## 10 · Real data — Breast Cancer Wisconsin

569 tumours, 30 features, bundled with sklearn — no download needed.

**TODO 13** — run the same five steps. First fit `LogisticRegression(max_iter=1000)` directly (expect a
`ConvergenceWarning`), report accuracy. Then fit the same model inside a
`make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))` and report accuracy again. Which
converges cleanly? Which scores higher?


In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
Xc, yc = data.data, data.target
print(Xc.shape, list(data.target_names))


In [ ]:
# TODO 13


**TODO 14** — using the **scaled** pipeline's coefficients (`pipe.named_steps['logisticregression'].coef_`),
print the 10 largest by absolute value, feature name next to coefficient.

Why would sorting the **unscaled** model's coefficients by size instead give a misleading ranking?
(Same trap as linear regression's California Housing coefficients.)


In [ ]:
# TODO 14


---
# Solutions

Try everything above first. Really.


In [ ]:
# --- 1 ---
print(df['passed'].value_counts())
print(df['hours_studied'].describe())
# 31 fail, 29 pass -- nicely balanced. describe() on a 0/1 column would give a
# "mean" of 0.483, which is not a meaningful summary of a category.


In [ ]:
# --- 2 ---
plt.scatter(df['hours_studied'], df['passed'])
plt.xlabel('hours studied'); plt.ylabel('passed')
plt.show()
# rows overlap around 4.7-6.1 hours -- that's why a hard cutoff can't work perfectly


In [ ]:
# --- 3 ---
X = df[['hours_studied']]   # (60, 1)
y = df['passed']            # (60,)
print(X.shape, y.shape)


In [ ]:
# --- 4 ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)   # (48,1) (12,1) (48,) (12,)
print(y_test.value_counts())   # 6 / 6 -- stratify kept the ~50/50 ratio


In [ ]:
# --- 5 / 6 ---
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
print(model.coef_, model.intercept_)      # [[1.5574]]  [-7.7366]
boundary = -model.intercept_[0] / model.coef_[0][0]
print('boundary =', boundary)             # ~4.97 hours


In [ ]:
# --- 7 ---
print(model.predict_proba([[6.5]]))   # [[0.084, 0.916]]
print(model.predict([[6.5]]))         # [1]
preds = model.predict(X_test)


In [ ]:
# --- 8 ---
print(accuracy_score(y_test, preds))   # 0.917 -- 11 / 12 correct


In [ ]:
# --- 9 ---
print('train acc', model.score(X_train, y_train))
print('test  acc', accuracy_score(y_test, preds))
# 0.917 both -- exactly equal here. Not a rule (train is *usually* a little higher,
# since the model saw those rows already), but with a 1-feature model and only
# 48 training points there's little room for the model to overfit and pull ahead.


In [ ]:
# --- 10 ---
xs = np.linspace(0, 10, 200).reshape(-1, 1)
plt.scatter(df['hours_studied'], df['passed'], label='data')
plt.plot(xs, model.predict_proba(xs)[:, 1], color='red', label='P(passed)')
plt.axhline(0.5, color='gray', linestyle='--', linewidth=1)
plt.xlabel('hours studied'); plt.ylabel('P(passed)'); plt.legend()
plt.show()


In [ ]:
# --- 11 ---
w, b = 0.0, 0.0
lr = 0.3
for _ in range(20_000):
    p = sigmoid(w * x_all + b)
    w -= lr * np.mean((p - y_all) * x_all)
    b -= lr * np.mean(p - y_all)
print(w, b)                               # ~2.2124  -11.1581

check = LogisticRegression().fit(df[['hours_studied']], y_all)   # sklearn DEFAULTS
print(check.coef_, check.intercept_)      # ~[[1.643]] [-8.270] -- NOT the same!
# sklearn regularises by default (L2 penalty, C=1.0), which shrinks the weights.
# LogisticRegression(penalty=None).fit(...) would match our from-scratch numbers instead.


In [ ]:
# --- 12 ---
for rs in [0, 1, 2, 3, 4]:
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=rs, stratify=y)
    m = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
    acc = accuracy_score(yte, m.predict(Xte))
    print('stratified  ', rs, round(acc, 3), np.bincount(yte))

for rs in [0, 1, 2, 3, 4]:
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=rs)
    m = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
    acc = accuracy_score(yte, m.predict(Xte))
    print('unstratified', rs, round(acc, 3), np.bincount(yte))
# stratified test sets stay close to 6/6 every time.
# unstratified ones can drift (e.g. 9/3 at some seeds) -- and accuracy is noisier as a result.


In [ ]:
# --- 13 ---
Xtr, Xte, ytr, yte = train_test_split(Xc, yc, test_size=0.2, random_state=42, stratify=yc)

m1 = LogisticRegression(max_iter=1000)
m1.fit(Xtr, ytr)   # likely prints a ConvergenceWarning
print('unscaled', accuracy_score(yte, m1.predict(Xte)))   # 0.965

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
pipe.fit(Xtr, ytr)   # converges cleanly
print('scaled  ', accuracy_score(yte, pipe.predict(Xte)))   # 0.982


In [ ]:
# --- 14 ---
coefs = pd.Series(pipe.named_steps['logisticregression'].coef_[0], index=Xc.columns)
print(coefs.reindex(coefs.abs().sort_values(ascending=False).index).head(10))
# top features (worst texture, radius error, worst concave points, worst area, ...)
# are all negative: bigger/rougher tumour measurements push the prediction away from
# class 1 ("benign"), toward malignant -- matches medical intuition.
# unscaled coefficients would instead rank features by their raw units (e.g. "mean area"
# looks huge just because area is measured in the hundreds) -- not by real importance.
